## 0 Omnilingual Baseline

This notebook gets the baseline Word Error Rate (WER) and Character Error Rate (CER) scores for Meta's Omnilingual model.

In [2]:
# Cell 1: Imports
import torch
from tqdm import tqdm
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

In [3]:
# Cell 2: Configuration
MODEL_CARD = "omniASR_CTC_1B_v2"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

Device: cuda, Lang supported: True


In [4]:
# Cell 3: Load test data
df_test = load_all_data("test")
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


In [5]:
# Cell 4: Transcribe
pipeline = ASRInferencePipeline(model_card=MODEL_CARD, device=DEVICE)
audio_paths = df_test["audio_path"].tolist()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch = audio_paths[i:i+BATCH_SIZE]
    try:
        results = pipeline.transcribe(batch, lang=[LANGUAGE_CODE]*len(batch), batch_size=len(batch))
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at {i}: {e}")
        transcriptions.extend([""] * len(batch))

df_test["omnilingual_transcription"] = transcriptions

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

 39%|███▉      | 31/79 [00:29<00:36,  1.31it/s]

Batch error at 248: The map function has failed while processing the path 'data' of the input data. See nested exception for details.


100%|██████████| 79/79 [01:08<00:00,  1.16it/s]


In [6]:
# Cell 5: Compute metrics
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)


OVERALL RESULTS
Total test samples: 631
Word Error Rate (WER): 75.14%
Character Error Rate (CER): 26.03%

PER IDIOM RESULTS

CC-2021-05-28
  Samples: 81
  WER: 55.08%
  CER: 15.28%

RMPUTER-CC-2021-06-11
  Samples: 114
  WER: 73.82%
  CER: 23.51%

RMSURMIRAN-CC-2021-12-23
  Samples: 151
  WER: 81.98%
  CER: 30.49%

RMSURSILV-CC-2021-05-28
  Samples: 94
  WER: 75.75%
  CER: 24.65%

RMSUTSILV-CC-2022-05-18
  Samples: 94
  WER: 84.48%
  CER: 34.20%

RMVALLADER-CC-2021-05-28
  Samples: 97
  WER: 73.74%
  CER: 24.78%

SUMMARY TABLE
                   idiom  samples  wer_mean  wer_std  cer_mean  cer_std
           cc-2021-05-28       81     55.08    11.51     15.28     5.73
   rmputer-cc-2021-06-11      114     73.82    24.73     23.51    11.05
rmsurmiran-cc-2021-12-23      151     81.98    17.79     30.49    15.32
 rmsursilv-cc-2021-05-28       94     75.75    10.66     24.65     8.15
 rmsutsilv-cc-2022-05-18       94     84.48     9.35     34.20    14.47
rmvallader-cc-2021-05-28       97 

In [7]:
# Cell 6: Show examples
show_examples(df_test, "omnilingual_transcription")


EXAMPLE TRANSCRIPTIONS

Idiom: rmputer-cc-2021-06-11
Reference:    Dasper ils mürs vegnan eir chüros ils chastagners, actuelmaing dad üna gruppa da recruts civils chi appredschan la lavur our illa natüra.
Hypothesis:   daspera ls mürs vegnen er ciüros als ciastagners actuelmeinc da d üna grupa de recruts zivils ci apregian la lavur ori la natüra
WER: 81.8% | CER: 18.2%
----------------------------------------

Idiom: cc-2021-05-28
Reference:    e deblas vischnancas ch’èn mo pli posts executivs dal chantun. Il referendum duai esser in cler signal cunter adina dapli prescripziuns da surengiu e 
Hypothesis:   e deblas vishnanchas ch'en mopli posts executivs dal chantun il referendum duei esser in cler signal cunter adina da pli per scripziuns dasurengiu e p
WER: 38.6% | CER: 7.0%
----------------------------------------

Idiom: rmsurmiran-cc-2021-12-23
Reference:    I vagn er schon gia onns tgi vagn prest betg rabeglea ansemen las pistas, tg'ins ò igl davos gia 1 2 spurs anfignen ea.
Hyp